# Blender → Wan VACE (ComfyUI) + Test Veo 3 — chạy trên Google Colab

**Trước khi chạy:** `Runtime → Change runtime type → T4 GPU` (miễn phí) hoặc L4/A100 (Colab Pro).

| Phần | Nội dung | Chi phí |
|---|---|---|
| A | ComfyUI + Wan 2.1 VACE 1.3B (video-to-video giữ đúng không gian Blender) | Miễn phí (Colab T4) |
| B | Test xem Veo 3 có "hiểu" và giữ không gian từ ảnh render Blender | API Gemini có tính phí, **tách riêng** với tín dụng trong Flow/Gemini app |

Chạy lần lượt từng ô từ trên xuống.

## A1. Kiểm tra GPU + kết nối Google Drive (để lấy ảnh/khung hình Blender và lưu kết quả)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/blender-veo'   # đặt thư mục ai_export của Blender vào đây
os.makedirs(WORK, exist_ok=True)
print('Thư mục làm việc:', WORK)

## A2. Cài ComfyUI + node đọc/ghi video

In [ ]:
%cd /content
!git clone -q https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt
!git clone -q https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite custom_nodes/ComfyUI-VideoHelperSuite
!pip install -q -r custom_nodes/ComfyUI-VideoHelperSuite/requirements.txt
!git clone -q https://github.com/Fannovel16/comfyui_controlnet_aux custom_nodes/comfyui_controlnet_aux
!pip install -q -r custom_nodes/comfyui_controlnet_aux/requirements.txt
# Đưa input/output vào Google Drive để không mất khi Colab ngắt
!rm -rf input output
!mkdir -p {WORK}/comfy_input {WORK}/comfy_output
!ln -s {WORK}/comfy_input input
!ln -s {WORK}/comfy_output output
print('Đã cài xong ComfyUI')

## A3. Tải model Wan 2.1 VACE

- **1.3B** (mặc định): chạy được trên T4 miễn phí, 480p.
- **14B**: đẹp hơn nhiều nhưng cần A100/L4 (Colab Pro). Đổi `USE_14B = True`.

In [ ]:
USE_14B = False
BASE = 'https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files'
M = '/content/ComfyUI/models'
files = [
  (f'{BASE}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors', f'{M}/text_encoders'),
  (f'{BASE}/vae/wan_2.1_vae.safetensors',                          f'{M}/vae'),
  (f'{BASE}/diffusion_models/' + ('wan2.1_vace_14B_fp16.safetensors' if USE_14B
                                  else 'wan2.1_vace_1.3B_fp16.safetensors'), f'{M}/diffusion_models'),
]
!apt-get -qq install -y aria2 > /dev/null
for url, d in files:
    !aria2c -q -c -x 16 -s 16 -d {d} -o {url.split('/')[-1]} "{url}"
    print('OK', url.split('/')[-1])

## A4. Chạy ComfyUI và mở giao diện web (link trycloudflare.com)

In [ ]:
import subprocess, time, re
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
comfy = subprocess.Popen(['python','main.py','--listen','127.0.0.1','--port','8188'],
                         cwd='/content/ComfyUI', stdout=open('/content/comfy.log','w'), stderr=subprocess.STDOUT)
for _ in range(120):
    if 'To see the GUI go to' in open('/content/comfy.log').read(): break
    time.sleep(2)
tun = subprocess.Popen(['/content/cloudflared','tunnel','--url','http://127.0.0.1:8188'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tun.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        print('👉 Mở ComfyUI tại:', m.group(0)); break

## A5. Dùng trong ComfyUI (làm trong giao diện web)

1. Đưa file từ Blender (thư mục `ai_export/anim/`) vào `MyDrive/blender-veo/comfy_input/`:
   - `rgb/` = khung hình render chuyển động camera
   - `depth/` = depth map tương ứng
2. Trong ComfyUI: **Workflow → Browse Templates → Video → Wan VACE** (chọn mẫu *Control Video / Depth*).
3. Chọn đúng model đã tải: `wan2.1_vace_1.3B_fp16`, `umt5_xxl_fp8_e4m3fn_scaled`, `wan_2.1_vae`.
4. **Control video** = thư mục `depth/` (dùng node *Load Images (Path)* của VideoHelperSuite).
   **Reference image** = 1 ảnh từ `rgb/` hoặc keyframe Blender → giữ đúng màu, vật liệu, bố cục.
5. Kích thước 832×480, length 81 khung, 16 fps. Prompt mô tả độ chân thực:
   *"photorealistic, cinematic, natural daylight, real materials, film grain, 35mm lens"*.
6. Bấm **Run**. Trên T4 mất khoảng 10–20 phút/clip 5 giây. Kết quả nằm ở `comfy_output/`.

Vì camera và hình học đến từ depth map Blender, **không gian được giữ đúng 100%** — AI chỉ thay "lớp da" cho chân thực.

---
# B. Test: Veo 3 có hiểu không gian Blender không?

Có 2 cách test:
- **Cách 1 — miễn phí thêm, dùng tín dụng bạn đang có:** làm thủ công trong Google Flow theo bảng ở file `TEST-VEO3.md`.
- **Cách 2 — tự động qua API (ô bên dưới):** cần API key Gemini đã bật thanh toán (aistudio.google.com). Tính phí theo giây video, riêng với tín dụng Flow.

Đặt key vào **Colab Secrets** (biểu tượng 🔑 bên trái) với tên `GEMINI_API_KEY`.

In [ ]:
!pip install -q -U google-genai
import time, os
from google import genai
from google.genai import types
from google.colab import userdata
client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

# Tên model có thể đổi theo thời gian — xem danh sách tại ai.google.dev/gemini-api/docs/video
MODEL = 'veo-3.1-fast-generate-preview'   # bản Fast rẻ hơn để thử; bản đầy đủ: 'veo-3.1-generate-preview'

KF = f'{WORK}/ai_export/keyframes'        # thư mục keyframe Blender xuất ra
def img(path): return types.Image(image_bytes=open(path,'rb').read(), mime_type='image/png')

def veo(name, prompt, first, last=None):
    cfg = types.GenerateVideosConfig(aspect_ratio='16:9', last_frame=img(last) if last else None)
    op = client.models.generate_videos(model=MODEL, prompt=prompt, image=img(first), config=cfg)
    while not op.done:
        time.sleep(10); op = client.operations.get(op)
    v = op.response.generated_videos[0]
    client.files.download(file=v.video)
    out = f'{WORK}/veo_test_{name}.mp4'
    v.video.save(out); print('Đã lưu', out)
    return out

### Chạy 3 bài test (đổi tên camera cho đúng với cảnh của bạn)
- **T1** – chỉ khung đầu, camera đứng yên, có hành động → Veo có giữ phòng không?
- **T2** – khung đầu + khung cuối từ 2 camera khác nhau → Veo có "đi" đúng trong không gian không?
- **T3** – khung đầu, camera xoay 180° → kiểm tra Veo có tự bịa phần phòng chưa thấy không (bài khó nhất).

In [ ]:
veo('T1_tinh',  'Static camera. Soft daylight moves slowly across the room, curtains sway gently. Photorealistic, keep every object exactly in place.',
    f'{KF}/Cam_A.png')
veo('T2_dau_cuoi', 'Smooth slow dolly camera move through the same room, photorealistic, no new objects, consistent architecture.',
    f'{KF}/Cam_A.png', f'{KF}/Cam_B.png')
veo('T3_xoay', 'Camera slowly pans 180 degrees to reveal the rest of the room. Photorealistic.',
    f'{KF}/Cam_A.png')

Chấm điểm từng video theo bảng trong `TEST-VEO3.md`.